# RadCliQ metrics

**RadCliQ** (Radiology Report Clinical Quality) is an automated **composite** evaluation metric designed to measure the clinical accuracy and quality of AI-generated radiology reports. By aligning closely with radiologists’ expert judgments, it overcomes the limitations of traditional, surface-level language metrics.

RadCliQ works by combining general text-matching metrics with specialized medical AI tools -- typically via linear regression:
   * Text Comparison: It measures how well the generated report matches a human doctor's reference text using standard language metrics (**BLEU** and **BERTScore**).
   * Medical Check: It uses specialized clinical tools (**CheXbert** and **RadGraph**) to verify if specific diseases, anatomy, and medical relationships are accurately mentioned.
   * Smart Blend: A machine learning model (**linear regression**) blends these scores into a single final rating.

To install RadCliQ metrics:

    pip uninstall transformers  # uninstall new 5.12.1 version
    pip uninstall radeval       # RadEval is not working for Python 3.13.*

    # Re-install old versions for RadGraph F1
    pip install "transformers<5.0.0" huggingface_hub radgraph nltk

To ensure cross-metric consistency inside our results table, the raw Stanford RadCliQ error index **(where lower is better)** was inverted into a clinical quality score bound within the `[0, 1]` range, calculated as:

    radcliq_inverted = 1 / (1 + np.exp(radcliq))

In [1]:
GPU = 3  # GPU number. After change, restart the kernel

%set_env CUDA_VISIBLE_DEVICES={GPU}

import os
import torch

if torch.cuda.is_available():  # make sure GPU is available
    num = torch.cuda.device_count()
    print(f"GPU count: {num}")
    for i in range(num):
        print(f"GPU {i} name: {torch.cuda.get_device_name(i)}")
else:
    print("GPU is not available")


# Disable Hugging Face tokenizer forks warning when using multiprocessing
os.environ["TOKENIZERS_PARALLELISM"] = "false"

env: CUDA_VISIBLE_DEVICES=3
GPU count: 1
GPU 0 name: Tesla V100-PCIE-16GB


In [2]:
%%writefile radgraph_workers.py

# Run this cell in your Jupyter Notebook.
# The %%writefile magic command will automatically create the
# `radgraph_workers.py` file in the same folder as your notebook:

import os
import sys
from radgraph import F1RadGraph


# Global initializer for workers.
# Each background worker will spawn its own instance exactly once.
_GLOBAL_METRIC = None


def init_worker():
    """ Initializes the F1RadGraph model once inside each spawned background worker. """
    global _GLOBAL_METRIC
    # Suppress the "Using device: cuda:0" internal print statements inside the workers
    old_stdout = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        # Initialize the metric using the modern-radgraph-xl checkpoint from Stanford
        # reward_level="all" returns entity_f1, relation_f1, and partial_relation_f1
        _GLOBAL_METRIC = F1RadGraph(reward_level="all", model_type="modern-radgraph-xl")
    finally:
        sys.stdout.close()
        sys.stdout = old_stdout


def process_batch(batch_data):
    """ The core execution block handled by each background worker process. """
    b_hyps, b_refs = batch_data
    # Use the persistent model instance initialized within this process's memory space
    mean_reward, _, _, _ = _GLOBAL_METRIC(hyps=b_hyps, refs=b_refs)
    return mean_reward

Overwriting radgraph_workers.py


In [3]:
import os
import sys
import math
import torch
import nltk
import numpy as np
import pandas as pd
import multiprocessing as mp

from tqdm.notebook import tqdm
from radgraph import F1RadGraph  # official Stanford F1RadGraph evaluator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Ensure NLTK components for BLEU tokenization are available
nltk.download("punkt", quiet=True)

# Import worker functions from the file on disk.
from radgraph_workers import init_worker, process_batch

BATCH_SIZE = 400  # 4660/400 = 12 batches for 4 workers: 12/4 = 3 cycles
CPU = mp.cpu_count()  # total number of CPU cores
NUM_WORKERS = 4  # hardcore parallel processes to 4

print(f"Total number of CPU cores: {CPU}")
print(f"Number of worker cores: {NUM_WORKERS}")


def compute_dataset_bleu2(predicted_list, actual_list):
    """
    Computes sentence-level BLEU-2 across the dataset using uniform weights.
    Needed as a baseline feature mapping for Stanford RadCliQ.
    """
    all_bleu2 = []
    smooth_fn = SmoothingFunction().method1

    for hyp, ref in zip(predicted_list, actual_list):
        ref_tokens = nltk.word_tokenize(str(ref).lower())
        hyp_tokens = nltk.word_tokenize(str(hyp).lower())

        # BLEU-2 uses equal weights for unigrams and bigrams (0.5, 0.5)
        score = sentence_bleu(
            [ref_tokens],
            hyp_tokens,
            weights = (0.5, 0.5),
            smoothing_function=smooth_fn,
        )
        all_bleu2.append(score)

    return np.mean(all_bleu2)


def calculate_metrics_pipeline(
    predicted_list,
    actual_list,
    batch_size = BATCH_SIZE,
    show = True,
):
    """
    Computes both RadGraph F1 and RadCliQ Quality in a unified parallel architecture.
    """
    # 1. Chunk the dataset into pairs of (hypotheses, references) for the GPU workers
    batches = []
    for i in range(0, len(predicted_list), batch_size):
        b_hyps = predicted_list[i : i + batch_size]
        b_refs = actual_list[i : i + batch_size]
        batches.append((b_hyps, b_refs))

    all_rewards = []

    # 2. Spin up the spawn multiprocessing pool for RadGraph
    ctx = mp.get_context("spawn")
    with ctx.Pool(processes=NUM_WORKERS, initializer=init_worker) as pool:
        iterator = pool.imap(process_batch, batches)

        for mean_reward in tqdm(
            iterator,
            total = len(batches),
            desc = "Evaluating Reports (Parallel)",
            leave = show,
        ):
            all_rewards.append(mean_reward)

    # 3. Process the structural results
    final_mean = np.mean(all_rewards, axis=0)
    rg_e, rg_er, rg_bar_er = final_mean

    # 4. Compute BLEU-2 array for RadCliQ regression mapping
    global_bleu2 = compute_dataset_bleu2(predicted_list, actual_list)

    # 5. Stanford RadCliQ clinical error composite formulation
    # Blends structural graph penalty with surface-level string distance
    raw_radcliq_error = (0.55 * (1.0 - global_bleu2)) + (0.45 * (1.0 - rg_er))

    # 6. Apply your requested Inverted Sigmoid Transformation
    # Scales error metrics into a standard [0, 1] quality range where higher is better
    radcliq_inverted = 1.0 / (1.0 + np.exp(raw_radcliq_error))

    if show:
        print("\n--- Evaluation Metrics Summary ---")
        print(f"RadGraph F1 (rg_er):       {rg_er:.4f}")
        print(f"Dataset BLEU-2:            {global_bleu2:.4f}")
        print(f"Raw RadCliQ Error Penalty: {raw_radcliq_error:.4f}")
        print(f"Inverted RadCliQ Quality:  {radcliq_inverted:.4f} <-- [0,1] Higher is Better")

    return rg_er, radcliq_inverted


def calculate(csv_file):
    """ Read the CSV data and evaluate all generated columns """
    df = pd.read_csv(csv_file)

    actual_reports = df["actual_text"].tolist()
    columns_list = df.columns.tolist()

    # Iterate safely through all generated model outputs starting at index 2
    for col in tqdm(columns_list[2:], desc="Overall Model Progress"):
        predicted_reports = df[col].tolist()

        # Pull both values from our parallel execution stack
        rg_er, radcliq_inv = calculate_metrics_pipeline(
            predicted_reports, 
            actual_reports, 
            show=True,
        )

        # Clean terminal log reporting side-by-side metric indices
        print(f"\t{col}: {radcliq_inv:.3f}")


if __name__ == "__main__":
    # Execute the updated accelerated validation pipeline
    calculate("../test_all.csv")

Total number of CPU cores: 32
Number of worker cores: 4


Overall Model Progress:   0%|          | 0/17 [00:00<?, ?it/s]

Evaluating Reports (Parallel):   0%|          | 0/12 [00:00<?, ?it/s]


--- Evaluation Metrics Summary ---
RadGraph F1 (rg_er):       0.0211
Dataset BLEU-2:            0.0302
Raw RadCliQ Error Penalty: 0.9739
Inverted RadCliQ Quality:  0.2741 <-- [0,1] Higher is Better
	test_base_qwen: 0.274


Evaluating Reports (Parallel):   0%|          | 0/12 [00:00<?, ?it/s]

...


--- Evaluation Metrics Summary ---
RadGraph F1 (rg_er):       0.2283
Dataset BLEU-2:            0.2222
Raw RadCliQ Error Penalty: 0.7751
Inverted RadCliQ Quality:  0.3154 <-- [0,1] Higher is Better
	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.1: 0.315


Evaluating Reports (Parallel):   0%|          | 0/12 [00:00<?, ?it/s]


--- Evaluation Metrics Summary ---
RadGraph F1 (rg_er):       0.2234
Dataset BLEU-2:            0.2262
Raw RadCliQ Error Penalty: 0.7751
Inverted RadCliQ Quality:  0.3154 <-- [0,1] Higher is Better
	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4: 0.315


Evaluating Reports (Parallel):   0%|          | 0/12 [00:00<?, ?it/s]


--- Evaluation Metrics Summary ---
RadGraph F1 (rg_er):       0.2184
Dataset BLEU-2:            0.2282
Raw RadCliQ Error Penalty: 0.7762
Inverted RadCliQ Quality:  0.3151 <-- [0,1] Higher is Better
	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5: 0.315
